# 02 · Fine-tune with Unsloth QLoRA

Trains Qwen2.5-3B on the tool-calling task. ~30–60 min on a free T4. Run notebook 01 first (it writes the JSONL splits).

In [ ]:
import os, sys, subprocess
REPO_URL = "https://github.com/Niikhi/peft-function-calling.git"
REPO_DIR = "peft-function-calling"
if not os.path.exists("src") and not os.path.exists(os.path.join(REPO_DIR, "src")):
    subprocess.run(["git", "clone", REPO_URL], check=True)
if os.path.exists(os.path.join(REPO_DIR, "src")):
    os.chdir(REPO_DIR)
sys.path.insert(0, os.getcwd())
print("Working dir:", os.getcwd())


In [ ]:
!pip -q install unsloth

In [ ]:
import os
os.environ["HF_HUB_REPO_ID"] = "Nikrobber/qwen2.5-3b-tool-calling"
from huggingface_hub import notebook_login
notebook_login()


In [ ]:
from src.config import load_config
from src.data import build_text_dataset, load_jsonl
import os

cfg = load_config()

if not os.path.exists(cfg.file_in("data_dir", "train.jsonl")):
    from src.data import load_records, split_records, save_jsonl
    for name, recs in split_records(cfg, load_records(cfg)).items():
        save_jsonl(recs, cfg.file_in("data_dir", f"{name}.jsonl"))


In [ ]:
from src.model import load_model_and_tokenizer, add_lora

model, tokenizer = load_model_and_tokenizer(cfg)
model = add_lora(cfg, model)

In [ ]:
train_ds = build_text_dataset(load_jsonl(cfg.file_in("data_dir", "train.jsonl")), tokenizer)
eval_ds  = build_text_dataset(load_jsonl(cfg.file_in("data_dir", "val.jsonl")), tokenizer)
print(train_ds, eval_ds)

In [ ]:
from src.train import build_trainer, save_log_history, save_adapter, train_resumable

trainer = build_trainer(cfg, model, tokenizer, train_ds, eval_ds)
train_resumable(cfg, trainer)  # resumes from outputs/ checkpoint if one exists

log_path = save_log_history(cfg, trainer)
save_adapter(cfg, model, tokenizer)

### Loss curves

In [ ]:
from src.visualize import plot_loss_curves, plot_lr_schedule
from IPython.display import Image
plot_lr_schedule(log_path, cfg.file_in("figures_dir", "lr_schedule.png"))
Image(plot_loss_curves(log_path, cfg.file_in("figures_dir", "loss_curves.png")))